In [63]:
import numpy as np
import pandas as pd

In [64]:
df = pd.read_csv("../data/data_cleaned-v6.csv")

In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29013 entries, 0 to 29012
Data columns (total 31 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Price                            29013 non-null  float64
 1   new_car_price                    26143 non-null  float64
 2   overview_Fuel type               28152 non-null  object 
 3   spec_Regenerative Braking        3304 non-null   object 
 4   spec_CNG Tank Capacity (L/Kg)    179 non-null    object 
 5   spec_Turbocharger/ Supercharger  24958 non-null  object 
 6   spec_Four-Wheel-Drive            24843 non-null  object 
 7   Manufacturing Year               29013 non-null  int64  
 8   Brand                            29013 non-null  object 
 9   Model                            29013 non-null  object 
 10  Varient                          29013 non-null  object 
 11  Kilometer                        28152 non-null  float64
 12  ownership         

In [66]:
def clean(val):
    try:
        return "No" if "No" in val else "Yes"
    except:
        return "No"


In [67]:
df["Regenerative_Braking"] = df["spec_Regenerative Braking"].apply(
    clean
)

In [68]:
df.drop(columns=["spec_Regenerative Braking"], inplace=True)

In [69]:
def clean_cng_capacity(val):
    try:
        return val.split(" ")[-2]
    except:
        return val


df["CNG_Capacity"] = (
    df["spec_CNG Tank Capacity (L/Kg)"].apply(clean_cng_capacity).fillna(0)
)

In [70]:
df.drop(columns=["spec_CNG Tank Capacity (L/Kg)"], inplace=True)

In [71]:
df["Supercharger"] = df["spec_Turbocharger/ Supercharger"].apply(clean)

In [72]:
df.drop(columns=["spec_Turbocharger/ Supercharger"], inplace=True)

In [73]:

df["Four_Wheel_Drive"] = df["spec_Four-Wheel-Drive"].apply(clean)

In [74]:
df.drop(columns=["spec_Four-Wheel-Drive"],inplace=True)

In [75]:
df.Four_Wheel_Drive.value_counts()

Four_Wheel_Drive
No     26363
Yes     2650
Name: count, dtype: int64

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29013 entries, 0 to 29012
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Price                        29013 non-null  float64
 1   new_car_price                26143 non-null  float64
 2   overview_Fuel type           28152 non-null  object 
 3   Manufacturing Year           29013 non-null  int64  
 4   Brand                        29013 non-null  object 
 5   Model                        29013 non-null  object 
 6   Varient                      29013 non-null  object 
 7   Kilometer                    28152 non-null  float64
 8   ownership                    27800 non-null  object 
 9   Transmission                 28152 non-null  object 
 10  Insurance                    29013 non-null  object 
 11  Registration_Type            28152 non-null  object 
 12  City                         28152 non-null  object 
 13  Engine(cc)      

In [77]:
df[df["new_car_price"].isna()]["Brand"].value_counts()

Brand
Hyundai          406
Maruti           394
Mahindra         281
Mercedes-Benz    267
Honda            232
Toyota           217
Renault          172
Tata             144
Volkswagen       113
Kia               89
Audi              85
BMW               73
Jeep              69
Ford              61
Skoda             60
Land              54
Chevrolet         44
Volvo             35
MG                34
Jaguar            16
Fiat              12
Nissan            12
Name: count, dtype: int64

In [78]:
brand_wise_np_median = {}
for brand in df.Brand.unique():
    brand_wise_np_median[brand] = df[df.Brand == brand]["new_car_price"].median()

In [79]:
def fill_np(row):
    if str(row["new_car_price"]) == "nan":
        return brand_wise_np_median[row["Brand"]]
    return row["new_car_price"]


df["new_car_price"] = df.apply(fill_np, axis=1)

In [80]:
df = df[~df["overview_Fuel type"].isna()]

In [81]:
df["ownership"] = df["ownership"].fillna("1")

In [82]:
df["Transmission"] = df["Transmission"].replace("Not","Manual")

In [83]:
binary_col = ["Insurance"]

In [84]:
df["Registration_Type"] = df["Registration_Type"].replace("Not Available","Individual")

In [85]:
df["Engine(cc)"] = df["Engine(cc)"].str.strip()

In [86]:
df["Engine(cc)"] = df["Engine(cc)"].replace("Not Applicable", np.nan)

In [87]:
def clean_engine(val):
    try:
        if "Cylinders" in val:
            return np.nan
        return val
    except:
        return val

In [88]:
df["Engine(cc)"] = df["Engine(cc)"].apply(clean_engine).astype("float")

In [89]:
df.groupby(["Model"])["Engine(cc)"].median()

Model
1-Series    1995.0
1600           NaN
2           1998.0
3           1995.0
3-Series    1995.0
             ...  
i20         1197.0
i4             NaN
i5             NaN
i7             NaN
iX             NaN
Name: Engine(cc), Length: 275, dtype: float64

In [90]:
def fill_engine(row):
    if str(row["Engine(cc)"]) == "nan":
        car_model = row["Model"]
        return df[df.Model == car_model]["Engine(cc)"].median()
    return row["Engine(cc)"]


df["Engine(cc)"] = df.apply(fill_engine, axis=1)

C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312

In [91]:
df[df["Engine(cc)"].isna()]["Model"].value_counts().index
# {"XUV400": 1950, "ZS": 1500}

Index(['XUV400', 'ZS', 'Windsor', 'W110', 'XEV', 'Be', 'Comet', 'i7', 'Jeep',
       'EQS', 'iX', 'EV6', 'EQC', 'Cyberster', 'EQB', 'EQE', 'Cresta', 'i5',
       'e2o', 'Fleetline', 'I-Pace', 'Genesis', 'e-tron', 'Kona', 'Concerto',
       'Commuter', 'EQA', 'M9', 'EX40', 'Tavera', '1600', '7HM', 'Ioniq',
       'MB-Class', 'i4', 'G580', 'Armada', 'Kaiser', 'Crown', 'EC40', 'Patrol',
       'Teana', 'Cedric', 'Impala', 'Alphard', 'CLS', 'Tourer', 'Megane',
       'Estate', 'Mark', 'Marshal'],
      dtype='object', name='Model')

In [92]:
df.isna().sum()[df.isna().sum() != 0]

Engine(cc)            289
Max_Power(bhp)       1013
Mileage              3385
Seating_Capacity      483
Fuel_Tank_Capcity    1283
dtype: int64

In [93]:
engine_cc_dict = {
    "XUV400": 1497,
    "ZS": 1498,
    "Windsor": 1498,
    "W110": 2179,
    "Be": 1497,
    "XEV": 1497,
    "Comet": 1198,
    "i7": 2998,
    "iX": 2998,
    "Jeep": 1956,
    "EQS": 2999,
    "EV6": 2199,
    "EQB": 1991,
    "EQC": 1991,
    "Cyberster": 1998,
    "Cresta": 1998,
    "EQE": 1991,
    "i5": 2998,
    "I-Pace": 1999,
    "e2o": 998,
    "Fleetline": 1498,
    "Genesis": 3470,
    "e-tron": 2995,
    "Commuter": 2755,
    "Concerto": 1590,
    "EQA": 1332,
    "M9": 1998,
    "Kona": 1591,
    "EX40": 1969,
    "7HM": 2494,
    "Tavera": 2499,
    "1600": 1600,
    "Ioniq": 1580,
    "MB-Class": 2143,
    "EC40": 1969,
    "i4": 1998,
    "Crown": 2494,
    "Kaiser": 2260,
    "Armada": 5552,
    "G580": 3982,
    "Cedric": 2960,
    "Patrol": 5552,
    "Teana": 2495,
    "Alphard": 2494,
    "Impala": 3600,
    "Tourer": 2491,
    "CLS": 2987,
    "Megane": 1498,
    "Estate": 1998,
    "Mark": 2491,
    "Marshal": 2498,
}

In [94]:
def fill_engine_2(row):
    if str(row["Engine(cc)"]) == "nan":
        model = row["Model"]
        if model in engine_cc_dict:
            return engine_cc_dict[model]
        return 0
    return row["Engine(cc)"]
df["Engine(cc)"] = df.apply(fill_engine_2,axis=1)

In [95]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28152 entries, 0 to 28151
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Price                        28152 non-null  float64
 1   new_car_price                28152 non-null  float64
 2   overview_Fuel type           28152 non-null  object 
 3   Manufacturing Year           28152 non-null  int64  
 4   Brand                        28152 non-null  object 
 5   Model                        28152 non-null  object 
 6   Varient                      28152 non-null  object 
 7   Kilometer                    28152 non-null  float64
 8   ownership                    28152 non-null  object 
 9   Transmission                 28152 non-null  object 
 10  Insurance                    28152 non-null  object 
 11  Registration_Type            28152 non-null  object 
 12  City                         28152 non-null  object 
 13  Engine(cc)           

In [96]:
df["Max_Power(bhp)"] = df["Max_Power(bhp)"].astype("float")

In [97]:
df["Max_Power(bhp)"].isna().sum()

np.int64(1013)

In [98]:
df["Max_Power(bhp)"].median()

np.float64(113.0)

In [99]:
def fill_max_power(row):
    if str(row["Max_Power(bhp)"]) == "nan":
        model = row["Model"]
        return df[df["Model"] == model]["Max_Power(bhp)"].median()
    return row["Max_Power(bhp)"]
df["Max_Power(bhp)"] = df.apply(fill_max_power,axis=1)

C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312

In [100]:
df[df["Max_Power(bhp)"].isna()]["Model"].unique()

array(['iX', 'i7', 'Jeep', 'Genesis', 'XUV400', 'W110', 'Windsor', 'ZS',
       'e-tron', 'I-Pace', 'EQB', 'EQC', 'EQS', 'MB-Class', 'Comet',
       'Cyberster', 'Commuter', 'Cresta', '7HM', 'i4', 'Tavera',
       'Concerto', 'EQE', 'Crown', 'Kaiser', 'Armada', 'EQA', 'G580',
       'M9', 'Cedric', 'Patrol', 'Teana', '1600', 'Alphard', 'Fleetline',
       'Impala', 'Tourer', 'Mondeo', 'Kona', 'CLS', 'Megane', 'Estate',
       'Mark', 'EX40', 'Marshal'], dtype=object)

In [101]:
max_power_estimate = {
    "iX": 516,          # BMW iX xDrive40/50 (most common ~516 bhp)
    "i7": 536,          # BMW i7 xDrive60
    "Jeep": 170,        # Jeep Compass 2.0 Diesel
    "Genesis": 300,     # Genesis sedan/SUV average
    "XUV400": 148,      # Mahindra XUV400 EV
    "W110": 48,         # Mercedes W110 190D
    "Windsor": 134,     # MG Windsor EV
    "ZS": 174,          # MG ZS EV
    "e-tron": 402,      # Audi e-tron 55
    "I-Pace": 394,      # Jaguar I-Pace
    "EQB": 288,         # Mercedes EQB 350
    "EQC": 402,         # Mercedes EQC
    "EQS": 516,         # Mercedes EQS 580
    "MB-Class": 120,    # Mercedes MB-Class (commercial van average)
    "Comet": 41,        # MG Comet EV
    "Cyberster": 503,   # MG Cyberster AWD
    "Commuter": 79,     # Toyota HiAce Commuter
    "Cresta": 100,      # Toyota Cresta average
    "7HM": 75,          # Hindustan 7HM truck/light commercial estimate
    "i4": 335,          # BMW i4 eDrive40
    "Tavera": 80,       # Chevrolet Tavera 2.5D
    "Concerto": 90,     # Honda Concerto
    "EQE": 288,         # Mercedes EQE 350+
    "Crown": 220,       # Toyota Crown hybrid average
    "Kaiser": 90,       # Kaiser Jeep average
    "Armada": 72,       # Nissan Armada (India old Mahindra Armada)
    "EQA": 188,         # Mercedes EQA 250+
    "G580": 579,        # Mercedes G580 EQ
    "M9": 241,          # Maxus MIFA 9 / M9 EV
    "Cedric": 200,      # Nissan Cedric average
    "Patrol": 275,      # Nissan Patrol 4.8L
    "Teana": 170,       # Nissan Teana 2.5
    "1600": 60,         # Datsun/Nissan 1600 class
    "Alphard": 247,     # Toyota Alphard Hybrid
    "Fleetline": 135,   # Chevrolet Fleetline
    "Impala": 250,      # Chevrolet Impala average
    "Tourer": 177,      # Honda Accord Tourer / similar average
    "Mondeo": 138,      # Ford Mondeo 2.0
    "Kona": 134,        # Hyundai Kona Electric
    "CLS": 255,         # Mercedes CLS 300d/350 average
    "Megane": 108,      # Renault Megane average
    "Estate": 75,       # Tata Estate
    "Mark": 220,        # Toyota Mark II average
    "EX40": 402,        # Volvo EX40 Twin Motor
    "Marshal": 62       # Mahindra Marshal
}

In [102]:
def fill_max_power_2(row):
    if str(row["Max_Power(bhp)"]) == "nan":
        return max_power_estimate[row["Model"]]
    return row["Max_Power(bhp)"]
df["Max_Power(bhp)"] = df.apply(fill_max_power_2,axis=1)

In [104]:
df["Mileage"].isna().sum()

np.int64(3385)

In [106]:
df["Mileage"].value_counts()

Mileage
15 kmpl       522
18 kmpl       496
17 kmpl       491
15.1 kmpl     369
18.6 kmpl     354
             ... 
10.42 kmpl      1
32.26 kmpl      1
19.86 kmpl      1
18.61 kmpl      1
13.93 kmpl      1
Name: count, Length: 654, dtype: int64

In [122]:
df["Mileage"] = df["Mileage"].str.replace("kmpl","").str.replace("km/kg","").str.strip().astype("float")

In [128]:
def fill_milage(row):
    if str(row["Mileage"] ) == "nan":
        model = row["Model"]
        return df[df["Model"] == model]["Mileage"].median()
    return row["Mileage"]
df["Mileage"] = df.apply(fill_milage,axis=1)

C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\LOQ\AppData\Roaming\Python\Python312

In [129]:
df[df["Mileage"].isna()]["Model"].unique()

array(['i5', 'iX', 'i7', 'X6', 'Jeep', 'Genesis', 'Ioniq', 'XUV400',
       'CLE', 'GL-Class', 'W110', 'Windsor', 'ZS', 'e-tron', 'I-Pace',
       'EV6', 'Be', 'e2o', 'XEV', 'EQB', 'EQC', 'EQS', 'GLB', 'MB-Class',
       'Comet', 'Cyberster', 'Commuter', 'Cresta', '7HM', 'Golf', 'EC40',
       'i4', 'Tavera', 'Concerto', 'EQE', 'Crown', 'XK', 'Kaiser',
       'Armada', 'EQA', 'G580', 'SLC', 'M9', 'Cedric', 'Patrol', 'Teana',
       'Hilux', '1600', 'Alphard', 'V90', 'Fleetline', 'Impala', 'Tourer',
       'Mondeo', 'CLS', 'Megane', 'Estate', 'Mark', 'EX40', 'Marshal'],
      dtype=object)

In [130]:
mileage_estimate = {
    "i5": 0.0,
    "iX": 0.0,
    "i7": 0.0,
    "X6": 10.5,
    "Jeep": 17.1,
    "Genesis": 12.5,
    "Ioniq": 0.0,
    "XUV400": 0.0,
    "CLE": 13.5,
    "GL-Class": 11.0,
    "W110": 13.0,
    "Windsor": 0.0,
    "ZS": 0.0,
    "e-tron": 0.0,
    "I-Pace": 0.0,
    "EV6": 0.0,
    "Be": 0.0,
    "e2o": 0.0,
    "XEV": 0.0,
    "EQB": 0.0,
    "EQC": 0.0,
    "EQS": 0.0,
    "GLB": 16.0,
    "MB-Class": 12.0,
    "Comet": 0.0,
    "Cyberster": 0.0,
    "Commuter": 12.0,
    "Cresta": 12.5,
    "7HM": 9.0,
    "Golf": 15.0,
    "EC40": 0.0,
    "i4": 0.0,
    "Tavera": 13.5,
    "Concerto": 13.0,
    "EQE": 0.0,
    "Crown": 18.0,
    "XK": 8.5,
    "Kaiser": 9.0,
    "Armada": 13.0,
    "EQA": 0.0,
    "G580": 0.0,
    "SLC": 14.0,
    "M9": 0.0,
    "Cedric": 10.0,
    "Patrol": 7.5,
    "Teana": 11.5,
    "Hilux": 12.5,
    "1600": 12.0,
    "Alphard": 15.0,
    "V90": 16.0,
    "Fleetline": 7.0,
    "Impala": 7.5,
    "Tourer": 15.5,
    "Mondeo": 14.0,
    "CLS": 15.0,
    "Megane": 16.0,
    "Estate": 14.0,
    "Mark": 12.0,
    "EX40": 0.0,
    "Marshal": 13.0
}

In [133]:
def fill_milage_2(row):
    if str(row["Mileage"]) == "nan":
        return mileage_estimate[row["Model"]]
    return row["Mileage"]
df["Mileage"] = df.apply(fill_milage_2,axis=1)

In [148]:
def clean_seating_cap(val):
    try:
        if "-" in val or "No" in val:
            return np.nan
        return val
    except:
        return val
df["Seating_Capacity"] = df["Seating_Capacity"].apply(clean_seating_cap)

In [151]:
df["Seating_Capacity"] = df["Seating_Capacity"].fillna(df["Seating_Capacity"].mode()[0]).astype("int")

In [154]:
df["Fuel_Tank_Capcity"].isna().sum()

np.int64(1283)

In [163]:
df["Fuel_Tank_Capcity"].unique()

array(['63', '70', '50', '54', '65', '75', '73', '55', '90', '62.4', '64',
       nan, '85', '58', '59', '57', '60', '68', '66', '78', '51', '67',
       '80', '83', '52', '45', '35', '38', '26', '42', '40', '60.9', '47',
       '37', '56', '32', '43', '62', '82', '66.5', '104', '86', '105',
       '27', '36', '48', '100', '93', '41', '28', '44', '15', '24', '71',
       '67.5', '72', '61', '74', '93.5', '-', '77', '76', '81', '89',
       '95', '69', '70.6', '110', '87', '26.2'], dtype=object)

In [171]:
df["Fuel_Tank_Capcity"] = df["Fuel_Tank_Capcity"].apply(lambda x: np.nan if "-" in str(x) else x).astype("float")

In [173]:
def fill_fuel_tank_cap(row):
    if str(row["Fuel_Tank_Capcity"]) == "nan":
        brand = row["Brand"]
        
        return df[df["Brand"] == brand]["Fuel_Tank_Capcity"].median()
    return row["Fuel_Tank_Capcity"]
df["Fuel_Tank_Capcity"] = df.apply(fill_fuel_tank_cap,axis=1)

In [178]:
(df.isna().sum() != 0).sum()

np.int64(0)

In [181]:
df.select_dtypes("object").info()

<class 'pandas.core.frame.DataFrame'>
Index: 28152 entries, 0 to 28151
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   overview_Fuel type           28152 non-null  object
 1   Brand                        28152 non-null  object
 2   Model                        28152 non-null  object
 3   Varient                      28152 non-null  object
 4   ownership                    28152 non-null  object
 5   Transmission                 28152 non-null  object
 6   Insurance                    28152 non-null  object
 7   Registration_Type            28152 non-null  object
 8   City                         28152 non-null  object
 9   Emission_Standard            28152 non-null  object
 10  Sunroof                      28152 non-null  object
 11  Praking_Sensors              28152 non-null  object
 12  Keyless_Start                28152 non-null  object
 13  GPS_Navigation               28152 n

In [193]:
binary_col = [
    "Four_Wheel_Drive",
    "Supercharger",
    "Regenerative_Braking",
    "Automatic_Emergency_Braking",
    "ADAS",
    "GPS_Navigation",
    "Keyless_Start",
    "Praking_Sensors",
    "Sunroof",
    "Insurance"
]

In [196]:
df[binary_col] = df[binary_col].replace({"Yes":1,"No":0})

C:\Users\LOQ\AppData\Local\Temp\ipykernel_15960\3632208760.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_col] = df[binary_col].replace({"Yes":1,"No":0})


In [187]:
df["CNG_Capacity"] = df["CNG_Capacity"].astype("float")

In [192]:
df["Driving_Range"] = df["Driving_Range"].str.replace("km","").str.strip().astype("float")

In [185]:
for col in df.columns:
    print(f"Showing value for {col} column")
    print(f"Dtype: {df[col].dtype}")
    print(f"Null Values: {df[col].isna().sum()}")
    print(f"Unqiue Values Count: {df[col].nunique()}")
    print(f"Unique Values: \n{df[col].value_counts()}")
    print("="*100)

Showing value for Price column
Dtype: float64
Null Values: 0
Unqiue Values Count: 2095
Unique Values: 
Price
500000.0     440
550000.0     380
450000.0     378
600000.0     370
650000.0     367
            ... 
1384000.0      1
872000.0       1
9095000.0      1
9950000.0      1
4999000.0      1
Name: count, Length: 2095, dtype: int64
Showing value for new_car_price column
Dtype: float64
Null Values: 0
Unqiue Values Count: 3942
Unique Values: 
new_car_price
1001000.0    414
787000.0     411
1789000.0    297
7588500.0    255
1064000.0    232
            ... 
7455000.0      1
4787000.0      1
3680000.0      1
5432000.0      1
4879000.0      1
Name: count, Length: 3942, dtype: int64
Showing value for overview_Fuel type column
Dtype: object
Null Values: 0
Unqiue Values Count: 6
Unique Values: 
overview_Fuel type
Petrol           16381
Diesel            9813
CNG               1009
Electric           847
LPG                 92
Not Available       10
Name: count, dtype: int64
Showing value for

In [197]:
df.select_dtypes("object").info()

<class 'pandas.core.frame.DataFrame'>
Index: 28152 entries, 0 to 28151
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   overview_Fuel type  28152 non-null  object
 1   Brand               28152 non-null  object
 2   Model               28152 non-null  object
 3   Varient             28152 non-null  object
 4   ownership           28152 non-null  object
 5   Transmission        28152 non-null  object
 6   Registration_Type   28152 non-null  object
 7   City                28152 non-null  object
 8   Emission_Standard   28152 non-null  object
dtypes: object(9)
memory usage: 2.1+ MB


In [198]:
df.to_csv("../data/data_cleaned-v7.csv")